In [ ]:
!pip install sympy math_verify pylatexenc vllm fastparquet

In [ ]:
!pip install flash-attn

In [ ]:
import os

os.environ["VLLM_USE_V1"] = "0"  # use V0 engine of vLLM. in V1 engine model weights cannot be directly access

In [ ]:
from transformers import PreTrainedModel, AutoModelForCausalLM, AutoTokenizer
from vllm.model_executor import set_random_seed as vllm_set_random_seed
from vllm import LLM, SamplingParams
from unittest.mock import patch
import torch
import numpy as np
import random
import pandas as pd
from tqdm import tqdm
from pandas import DataFrame
from tests.adapters import run_get_response_log_probs, run_tokenize_prompt_and_output, run_sft_microbatch_train_step


In [ ]:
def init_vllm(model_id: str, device: str, seed: int, gpu_memory_utilization: float = 0.85):
    """
    Start the inference process, here we use vLLM to hold a model on
    a GPU separate from the policy.
    """
    vllm_set_random_seed(seed)

    # Monkeypatch from TRL:
    # https://github.com/huggingface/trl/blob/22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py
    # Patch vLLM to make sure we can
    # (1) place the vLLM model on the desired device (world_size_patch) and
    # (2) avoid a test that is not designed for our setting (profiling_patch).
    world_size_patch = patch("torch.distributed.get_world_size", return_value=1)
    profiling_patch = patch(
    "vllm.worker.worker.Worker._assert_memory_footprint_increased_during_profiling",
    return_value=None
    )
    with world_size_patch, profiling_patch:
        return LLM(
            model=model_id,
            device=device,
            dtype=torch.bfloat16,
            enable_prefix_caching=True,
            gpu_memory_utilization=gpu_memory_utilization,
        )

def load_policy_into_vllm_instance(policy: PreTrainedModel, llm: LLM):
    """
    Copied from https://github.com/huggingface/trl/blob/22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py#L670.
    """
    state_dict = policy.state_dict()
    llm_model = llm.llm_engine.model_executor.driver_worker.model_runner.model
    llm_model.load_weights(state_dict.items())

In [ ]:
model_id = 'Qwen/Qwen2.5-Math-1.5B'
device_0 = 'cuda:0'
device_1 = 'cuda:1'
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
device = torch.device("cuda")

num_epochs = 10
batch_size = 8
gradient_accumulation_steps = 4

llm = init_vllm(model_id, device_0, seed)

In [ ]:
question = "Simplify $(3-i)(6+2i)$."

# Sample prompts.
prompts = [
    f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>""",
]

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["\n"]
)

sampling_params.stop = ["</answer>"]
sampling_params.include_stop_str_in_output = True

# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.generate(prompts, sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")

In [ ]:
from drgrpo_grader import r1_zero_reward_fn
r1_zero_reward_fn(generated_text, '20')

In [ ]:
from typing import Callable

def evaluate_vllm(
  vllm_model: LLM,
  reward_fn: Callable[[str, str], dict[str, float]],
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) :
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)
  output_rewards = []

  for index, output in enumerate(outputs):
    generated_text = output.outputs[0].text
    rewards = reward_fn(generated_text, answers[index])
    output_rewards.append(rewards)

  return output_rewards

In [7]:
def eval_validation_set(df: DataFrame):
    total_format_reward = 0.
    total_answer_reward = 0.
    total_reward = 0.
    prompts = []
    answers = []

    for index, row in tqdm(enumerate(df.sample(frac=0.1).itertuples())):
        question = row.problem
        answer = row.solution
        prompt = f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
    User: {question}
    Assistant: <think>"""
        prompts.append(prompt)
        answers.append(answer)

    rewards = evaluate_vllm(llm, r1_zero_reward_fn, prompts, answers, sampling_params)
    for reward in rewards:
        if reward['format_reward'] == 1.0:
            total_format_reward += 1
        if reward['answer_reward'] == 1.0:
            total_answer_reward += 1
        if reward['reward'] == 1.0:
            total_reward += 1

    print(f"Total format reward: {total_format_reward}")
    print(f"Total answer reward: {total_answer_reward}")
    print(f"Total reward: {total_reward}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2'
).to(device_1)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [2]:
df = pd.read_parquet("short_reasoning_86.parquet")
prompts = df['prompt'].tolist()
responses = df['response'].tolist()
tokenized = run_tokenize_prompt_and_output(prompts, responses, tokenizer)
input_ids = tokenized['input_ids'].long().to(device_1)
labels = tokenized['labels'].long().to(device_1)
response_mask = tokenized['response_mask'].to(device_1)

In [ ]:
from torch.utils.data import DataLoader, Dataset

class MathSFTDataset(Dataset):
    def __init__(self, input_ids, labels, masks):
        self.input_ids = input_ids
        self.labels = labels
        self.masks = masks

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        input_id = self.input_ids[index]
        label = self.labels[index]
        mask = self.masks[index]
        return input_id, label, mask

train_loader = DataLoader(
    dataset=MathSFTDataset(input_ids, labels, response_mask),
    batch_size=batch_size
)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

In [ ]:
global_step = -1
valid_df = pd.read_parquet("math_12k.parquet")

for epoch in range(num_epochs):
    model.train()

    for idx, (input_batch, label_batch, mask) in tqdm(enumerate(train_loader)):
        log_probs = run_get_response_log_probs(model, input_batch, label_batch, False)['log_probs']
        loss, _ = run_sft_microbatch_train_step(log_probs, mask, gradient_accumulation_steps)
        global_step += 1

        if (idx + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        if (global_step + 1) % 20 == 0:
            print(f"Step {global_step:06d}: Train loss: {loss.cpu().item()}")
            # reload model into vllm and eval
            load_policy_into_vllm_instance(model, llm)
            eval_validation_set(valid_df)

In [ ]:
import os

output_dir = "sft_model_vllm_eval"
os.makedirs(output_dir, exist_ok=True)

print("saving the model and tokenizer...")
model.save_pretrained(save_directory=output_dir)
tokenizer.save_pretrained(save_directory=output_dir)